# CropGuard: train on YOUR 38-class dataset (Google Colab, free GPU)

Trains a plant-disease classifier on the `train/` and `valid/` folders you already have (38 classes, 14 crops), with a stronger pretrained backbone than before, an **honest** split, calibration, an unknown-image gate, an optional **field-photo test** (PlantDoc), and an export that drops straight into the CropGuard backend.

**Before you run**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Get your data into Colab (pick one in the config cell):
   * `DATA_SOURCE = 'kaggle'` (easiest): upload your `kaggle.json` when asked; downloads the Kaggle *New Plant Diseases Dataset (Augmented)* (38 classes, train/valid). Check the slug in the config cell if the download fails.
   * `DATA_SOURCE = 'drive'`: zip your local `data\disease_dataset` folder and put the zip on Google Drive (see the config cell for the Windows command).
3. First do a smoke test with `QUICK = True` (about 10 minutes). Then set it to `False` and run all (about 1 to 1.5 hours with the defaults).

**Why this is better than a plain fine-tune**
| Problem | What the notebook does |
|---|---|
| This dataset is *augmented*: flipped/rotated copies of one photo can sit in both `train` and `valid`, so validation looks perfect but means little | finds near-duplicates (hash that ignores flips/rotations), reports how many `valid` images leak, and re-splits by group |
| Lab photos only | background replacement, scale augmentation (close-ups), MixUp/CutMix; optional test and fine-tuning on real PlantDoc field photos |
| "94 % sure" that means nothing | temperature scaling, ECE, a confidence threshold for a target precision, an energy gate for unfamiliar images |
| Crops with only healthy images (blueberry, raspberry, soybean) | reported explicitly; the backend labels advice for them as generic |

In [ ]:
!nvidia-smi -L
!pip -q install timm onnx onnxruntime grad-cam scikit-learn matplotlib pandas kaggle

In [ ]:
import os, sys, json, random, re, time, math, shutil, zipfile, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt

# ----------------------------- CONFIG: change here -----------------------------
QUICK          = False     # True = 10-minute smoke test on a small subset (numbers are meaningless)
DATA_SOURCE    = 'kaggle'  # 'kaggle' | 'drive' | 'local'
KAGGLE_SLUG    = 'vipoooool/new-plant-diseases-dataset'   # 38-class augmented dataset; verify on kaggle.com if the download fails
# Drive option. On Windows, make the zip with:   tar -a -c -f disease_dataset.zip disease_dataset      (run inside E:\MINOR\cropguard\data)
DRIVE_ZIP      = '/content/drive/MyDrive/cropguard/disease_dataset.zip'
LOCAL_DIR      = '../data/disease_dataset'                 # for 'local' runs in VS Code
BACKBONE       = 'convnext_tiny.fb_in22k_ft_in1k'          # alternatives: 'efficientnet_b2.ra_in1k', 'resnet50.a1_in1k' (any CNN in timm)
IMG, BS        = 224, 64
EPOCHS_HEAD, EPOCHS_FT = 2, 8
LR_HEAD, LR_FT = 1e-3, 1e-4
USE_MIXUP      = True      # MixUp/CutMix: more robust to odd framing and backgrounds
HONEST_SPLIT   = True      # re-split by near-duplicate groups (strongly recommended for this dataset)
USE_PLANTDOC   = True      # real field photos: honest field test + field fine-tuning
OOD_CROPS      = []        # e.g. ['Squash'] : leave these crops OUT of training to measure unknown-crop rejection (experiment only)
TARGET_PRECISION = 0.90
# --------------------------------------------------------------------------------
IN_COLAB = 'google.colab' in sys.modules
WORK = Path('/content') if IN_COLAB else Path('work'); WORK.mkdir(exist_ok=True)
OUT = WORK / 'cropguard38_artifacts'; OUT.mkdir(exist_ok=True)
SEED = 42
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORKERS = 0 if os.name == 'nt' else 2
if QUICK: EPOCHS_HEAD, EPOCHS_FT, BS = 1, 1, 16
print('device:', dev, '| torch', torch.__version__)
if dev.type == 'cpu': print('WARNING: no GPU. Use a GPU runtime or keep QUICK = True.')

## 1. Data

In [ ]:
def find_split_dirs(base):
    found = {}
    for r, d, f in os.walk(base):
        n = Path(r).name.lower()
        if n in ('train', 'valid', 'val', 'validation') and len(d) >= 5:
            found[n] = Path(r)
    return found.get('train'), found.get('valid') or found.get('val') or found.get('validation')

if DATA_SOURCE == 'kaggle':
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if IN_COLAB and not kj.exists():
        from google.colab import files
        print('Upload your kaggle.json'); files.upload()
        kj.parent.mkdir(exist_ok=True); shutil.move('kaggle.json', kj); os.chmod(kj, 0o600)
    if not any((WORK / 'data').glob('*')):
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        api.dataset_download_files(KAGGLE_SLUG, path=str(WORK / 'data'), unzip=True, quiet=False)
    DATA_ROOT = WORK / 'data'
elif DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    if not any((WORK / 'data').glob('*')):
        assert Path(DRIVE_ZIP).exists(), f'{DRIVE_ZIP} not found: upload the zip to Drive first'
        print('unzipping ...')
        with zipfile.ZipFile(DRIVE_ZIP) as z: z.extractall(WORK / 'data')
    DATA_ROOT = WORK / 'data'
else:
    DATA_ROOT = Path(LOCAL_DIR)

TRAIN_DIR, VALID_DIR = find_split_dirs(DATA_ROOT)
assert TRAIN_DIR is not None, f'no train/ folder with class sub-folders under {DATA_ROOT}'
print('train:', TRAIN_DIR, '\nvalid:', VALID_DIR)

EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
def scan(root, split):
    return [(str(f), d.name, split) for d in sorted(p for p in Path(root).iterdir() if p.is_dir()) for f in d.iterdir() if f.suffix.lower() in EXT]
df = pd.DataFrame(scan(TRAIN_DIR, 'train') + (scan(VALID_DIR, 'valid') if VALID_DIR else []), columns=['path', 'raw', 'orig_split'])
print(len(df), 'images |', df.raw.nunique(), 'classes |', df.orig_split.value_counts().to_dict())

## 2. Class metadata (names, crops, Hindi crop names)

In [ ]:
def tok(s): return re.sub(r'[^A-Za-z0-9]+', '_', s).strip('_')
CROP_ALIAS = {'Corn_maize': 'Maize', 'Cherry_including_sour': 'Cherry', 'Pepper_bell': 'Pepper'}
CROP_HI = {'Tomato': 'टमाटर', 'Potato': 'आलू', 'Pepper': 'शिमला मिर्च', 'Maize': 'मक्का', 'Apple': 'सेब', 'Grape': 'अंगूर', 'Peach': 'आड़ू',
           'Cherry': 'चेरी', 'Strawberry': 'स्ट्रॉबेरी', 'Raspberry': 'रसभरी', 'Blueberry': 'ब्लूबेरी', 'Orange': 'संतरा', 'Squash': 'स्क्वैश (कद्दू वर्ग)', 'Soybean': 'सोयाबीन'}
DIS_HI = {'healthy': 'स्वस्थ', 'Bacterial_spot': 'जीवाणु धब्बा रोग', 'Early_blight': 'अगेती झुलसा', 'Late_blight': 'पछेती झुलसा', 'Leaf_Mold': 'पर्ण फफूंद (लीफ मोल्ड)',
          'Septoria_leaf_spot': 'सेप्टोरिया पत्ती धब्बा', 'Target_Spot': 'टारगेट स्पॉट (लक्ष्य धब्बा)', 'Tomato_mosaic_virus': 'मोज़ेक विषाणु',
          'Tomato_Yellow_Leaf_Curl_Virus': 'पीला पत्ती मोड़क विषाणु', 'Spider_mites_Two_spotted_spider_mite': 'मकड़ी माइट (लाल मकड़ी)',
          'Powdery_mildew': 'चूर्णिल आसिता (पाउडरी मिल्ड्यू)', 'Common_rust': 'सामान्य रतुआ (रस्ट)', 'Black_rot': 'काला सड़न रोग (ब्लैक रॉट)',
          'Apple_scab': 'सेब का स्कैब (पपड़ी रोग)', 'Cedar_apple_rust': 'सीडर एप्पल रस्ट', 'Northern_Leaf_Blight': 'उत्तरी पत्ती झुलसा',
          'Leaf_scorch': 'पत्ती झुलसा (लीफ स्कॉर्च)', 'Haunglongbing_Citrus_greening': 'सिट्रस ग्रीनिंग (हुआंगलोंगबिंग)', 'Esca_Black_Measles': 'एस्का (ब्लैक मीज़ल्स)',
          'Leaf_blight_Isariopsis_Leaf_Spot': 'पत्ती झुलसा (इसारियोप्सिस पत्ती धब्बा)', 'Cercospora_leaf_spot_Gray_leaf_spot': 'सर्कोस्पोरा / ग्रे पत्ती धब्बा'}
DIS_EN = {'Spider_mites_Two_spotted_spider_mite': 'Spider Mites', 'Haunglongbing_Citrus_greening': 'Citrus Greening (Huanglongbing)', 'Esca_Black_Measles': 'Esca (Black Measles)',
          'Leaf_blight_Isariopsis_Leaf_Spot': 'Leaf Blight (Isariopsis Leaf Spot)', 'Cercospora_leaf_spot_Gray_leaf_spot': 'Cercospora / Gray Leaf Spot',
          'Tomato_mosaic_virus': 'Mosaic Virus', 'Tomato_Yellow_Leaf_Curl_Virus': 'Yellow Leaf Curl Virus'}

def parse_class(raw):
    """-> (display crop name, disease token, crop token). The class ID keeps the original crop token (e.g. Pepper_bell) so it matches
    the advice files and the model labels exactly; only the DISPLAY name is shortened (Pepper, Maize, Cherry)."""
    crop_raw, dis = raw.split('___', 1)
    crop_tok = tok(crop_raw)
    return CROP_ALIAS.get(crop_tok, crop_tok), tok(dis), crop_tok

def build_meta(raw_classes):
    classes, by_crop = [], {}
    for raw in raw_classes:
        crop, dis, crop_tok = parse_class(raw)
        healthy = dis.lower() == 'healthy'
        c = {'id': f'{crop_tok}___{dis}', 'crop': {'en': crop, 'hi': CROP_HI.get(crop, crop)},
             'disease': {'en': DIS_EN.get(dis, dis.replace('_', ' ').capitalize() if dis != 'healthy' else 'Healthy'), 'hi': DIS_HI.get(dis, dis.replace('_', ' '))},
             'healthy': healthy}
        if crop not in CROP_HI or dis not in DIS_HI: c['needs_translation'] = True
        classes.append(c); by_crop.setdefault(crop, []).append(healthy)
    return classes, sorted(c for c, hs in by_crop.items() if all(hs))

df['crop'] = df.raw.map(lambda r: parse_class(r)[0])
if OOD_CROPS:
    ood_df = df[df.crop.isin(OOD_CROPS)].copy(); df = df[~df.crop.isin(OOD_CROPS)].reset_index(drop=True)
else:
    ood_df = df.iloc[0:0].copy()
if QUICK:
    df = df.groupby('raw', group_keys=False).apply(lambda g: g.sample(min(len(g), 40), random_state=SEED)).reset_index(drop=True)
RAW_CLASSES = sorted(df.raw.unique()); NC = len(RAW_CLASSES)
CLASS_IDX = {c: i for i, c in enumerate(RAW_CLASSES)}; df['y'] = df.raw.map(CLASS_IDX)
CLASSES_META, HEALTHY_ONLY = build_meta(RAW_CLASSES)
print(NC, 'classes |', df.crop.nunique(), 'crops')
print('crops with ONLY a healthy class (the model cannot detect diseases for them):', HEALTHY_ONLY or 'none')
display(df.groupby('crop').raw.nunique().rename('classes').to_frame().T)

## 3. Honest split
The Kaggle *augmented* dataset stores flipped/rotated/brightness-changed copies of the same photo. If the copies land in both `train` and `valid`, accuracy looks near-perfect but does not tell you how the model behaves on a new leaf. We hash every image so that flips and rotations give the same value, group near-duplicates, report the leak, and split by group.

In [ ]:
def d4_variants(a):
    out = []
    for k in range(4):
        r = np.rot90(a, k); out.append(r); out.append(r[:, ::-1])
    return out

def canon_hash(path):
    """64-bit hash that is the same for flips/rotations (and robust to brightness changes)."""
    try:
        im = Image.open(path); im.draft('L', (64, 64))
        g = np.asarray(im.convert('L').resize((16, 16), Image.BILINEAR), dtype=np.float32)
    except Exception:
        return 0
    best = None
    for v in d4_variants(g):
        s = v.reshape(8, 2, 8, 2).mean((1, 3))
        h = int(np.packbits((s > s.mean()).flatten()).view('>u8')[0])
        best = h if best is None else min(best, h)
    return best

_POP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def popcount(x): return _POP[np.ascontiguousarray(x).view(np.uint8).reshape(x.shape + (8,))].sum(-1)

def group_ids(labels, hashes, max_dist=4, max_bucket=400):
    """Union images of the same class whose hashes differ in <= max_dist bits. 5 chunks => any such pair shares a chunk exactly."""
    n = len(labels); parent = list(range(n)); H = np.array(hashes, dtype=np.uint64)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for sh, bits in zip([0, 13, 26, 39, 52], [13, 13, 13, 13, 12]):
        key = ((H >> np.uint64(sh)) & np.uint64((1 << bits) - 1)).astype(np.int64)
        tmp = pd.DataFrame({'l': np.asarray(labels), 'k': key, 'i': np.arange(n)})
        for _, g in tmp.groupby(['l', 'k'])['i']:
            idx = g.values
            if len(idx) < 2 or len(idx) > max_bucket: continue
            d = popcount(H[idx][:, None] ^ H[idx][None, :])
            for a, b in np.argwhere(np.triu(d <= max_dist, 1)):
                ra, rb = find(idx[a]), find(idx[b])
                if ra != rb: parent[ra] = rb
    return np.array([find(i) for i in range(n)])

t0 = time.time()
paths = df.path.tolist()
if os.name == 'nt' or len(paths) < 2000:
    hashes = [canon_hash(p) for p in paths]
else:
    from concurrent.futures import ProcessPoolExecutor
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2) as ex:
        hashes = list(ex.map(canon_hash, paths, chunksize=256))
df['group'] = group_ids(df.y.values, hashes)
print(f'hashing + grouping: {time.time()-t0:.0f}s | {df.group.nunique()} groups for {len(df)} images')

if df.orig_split.nunique() > 1:
    sp = df.groupby('group').orig_split.agg(set)
    leaky = set(sp[sp.map(lambda s: len(s) > 1)].index)
    v = df[df.orig_split == 'valid']
    LEAK_PCT = 100 * v.group.isin(leaky).mean()
    print(f'>>> {LEAK_PCT:.1f}% of the dataset\'s own `valid` images have a near-duplicate (flip/rotation/brightness copy) in `train`.')
else:
    LEAK_PCT = None

from sklearn.model_selection import StratifiedGroupKFold
if HONEST_SPLIT:
    k = 5 if QUICK else 10
    folds = [i for _, i in StratifiedGroupKFold(k, shuffle=True, random_state=SEED).split(df, df.y, df.group)]
    te, va, tr = df.iloc[folds[0]], df.iloc[folds[1]], df.iloc[np.concatenate(folds[2:])]
    assert not (set(tr.group) & set(te.group)) and not (set(tr.group) & set(va.group)), 'group leak'
else:
    tr, va = df[df.orig_split == 'train'], df[df.orig_split == 'valid']; te = va
print('train', len(tr), '| val', len(va), '| test', len(te))

## 4. Augmentation and loaders

In [ ]:
import torchvision.transforms as T
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class PadSquare:          # same as the backend: keep the WHOLE photo, grey padding
    def __call__(self, im):
        w, h = im.size
        if w == h: return im
        s = max(w, h); out = Image.new('RGB', (s, s), (128, 128, 128)); out.paste(im, ((s - w) // 2, (s - h) // 2)); return out

class BGReplace:          # paste the leaf on random soil/grass/grey textures (lab photos have plain backgrounds)
    def __init__(self, p=0.5): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        a = np.asarray(img).astype(np.float32) / 255
        sat = (a.max(2) - a.min(2)) / (a.max(2) + 1e-6)
        m = ((sat > .18) & (a.max(2) > .12)).astype(np.uint8) * 255
        m = np.asarray(Image.fromarray(m).filter(ImageFilter.GaussianBlur(2))).astype(np.float32)[..., None] / 255
        h, w = a.shape[:2]
        base = np.array(random.choice([[.35, .25, .17], [.25, .45, .2], [.6, .6, .6], [.5, .5, .5]])) + np.random.uniform(-.1, .1, 3)
        bg = np.ones((h, w, 3), np.float32) * base; yy, xx = np.ogrid[:h, :w]
        for _ in range(random.randint(8, 20)):
            cy, cx, r = random.randint(0, h), random.randint(0, w), random.randint(10, 60)
            bg[(yy - cy) ** 2 + (xx - cx) ** 2 < r * r] = np.clip(base + np.random.uniform(-.2, .2, 3), 0, 1)
        bg = np.asarray(Image.fromarray((bg * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(random.uniform(2, 8)))).astype(np.float32) / 255
        return Image.fromarray(((a * m + bg * (1 - m)) * 255).astype(np.uint8))

tf_train = T.Compose([PadSquare(), BGReplace(0.5), T.RandomResizedCrop(IMG, scale=(0.25, 1.0), ratio=(0.75, 1.33)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(0.2),
                      T.RandomRotation(25), T.ColorJitter(0.3, 0.3, 0.3, 0.05), T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.3), T.ToTensor(), T.Normalize(MEAN, STD)])
tf_eval = T.Compose([PadSquare(), T.Resize((IMG, IMG)), T.ToTensor(), T.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(self, frame, tf): self.p, self.y, self.tf = frame.path.tolist(), frame.y.tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]

def loader(frame, tf, shuffle):
    return DataLoader(DS(frame, tf), batch_size=BS, shuffle=shuffle, num_workers=WORKERS, pin_memory=True, drop_last=shuffle, persistent_workers=WORKERS > 0)

L_tr, L_va, L_te = loader(tr, tf_train, True), loader(va, tf_eval, False), loader(te, tf_eval, False)
fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for i, a in enumerate(ax.flat):
    x = tf_train(Image.open(tr.path.iloc[(i * 97) % len(tr)]).convert('RGB')) * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
    a.imshow(x.permute(1, 2, 0).clamp(0, 1)); a.axis('off')
plt.suptitle('Training augmentations'); plt.show()

## 5. Model and training

In [ ]:
import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy

@torch.no_grad()
def predict(model, dl):
    model.eval().to(dev); lg, ys = [], []
    for x, y in dl:
        with torch.autocast(device_type=dev.type, enabled=dev.type == 'cuda'):
            lg.append(model(x.to(dev, non_blocking=True)).float().cpu())
        ys.append(y)
    return torch.cat(lg), torch.cat(ys)

def fit(model, train_loader, epochs, lr, tag, mixup=False, keep_best=True):
    model.to(dev)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(1, epochs * len(train_loader)))
    scaler = torch.cuda.amp.GradScaler(enabled=dev.type == 'cuda')
    mix = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.6, switch_prob=0.5, label_smoothing=0.1, num_classes=NC) if mixup else None
    crit = SoftTargetCrossEntropy() if mixup else nn.CrossEntropyLoss(label_smoothing=0.1)
    hist, best = [], (-1, None)
    for ep in range(epochs):
        model.train(); t0 = time.time(); tl = n = 0
        for x, y in train_loader:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            if mix is not None: x, y = mix(x, y)
            with torch.autocast(device_type=dev.type, enabled=dev.type == 'cuda'):
                loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 2.0); scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item() * len(x); n += len(x)
        lg, ys = predict(model, L_va); va_acc = (lg.argmax(1) == ys).float().mean().item()
        hist.append(dict(epoch=len(hist) + 1, loss=tl / max(n, 1), val_acc=va_acc))
        if va_acc > best[0]: best = (va_acc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        print(f'[{tag}] epoch {ep+1}/{epochs}  loss {tl/max(n,1):.4f}  val_acc {va_acc:.4f}  ({time.time()-t0:.0f}s)', flush=True)
    if keep_best and best[1] is not None: model.load_state_dict(best[1])
    return hist

seed_all()
model = timm.create_model(BACKBONE, pretrained=True, num_classes=NC)
print(BACKBONE, '| parameters (M):', round(sum(p.numel() for p in model.parameters()) / 1e6, 1))
head = list(model.get_classifier().parameters())
for p in model.parameters(): p.requires_grad = False
for p in head: p.requires_grad = True
h1 = fit(model, L_tr, EPOCHS_HEAD, LR_HEAD, 'head')
for p in model.parameters(): p.requires_grad = True
h2 = fit(model, L_tr, EPOCHS_FT, LR_FT, 'finetune', mixup=USE_MIXUP)
torch.save(model.state_dict(), OUT / 'lab_model.pt')
hist = pd.DataFrame(h1 + h2); hist['epoch'] = range(1, len(hist) + 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5)); hist.plot(x='epoch', y='loss', ax=ax[0], title='Train loss'); hist.plot(x='epoch', y='val_acc', ax=ax[1], title='Validation accuracy'); plt.show()

## 6. Evaluation on the held-out test split

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
def metrics(lg, ys):
    p = lg.argmax(1).numpy(); y = ys.numpy()
    return dict(accuracy=accuracy_score(y, p), macro_f1=f1_score(y, p, average='macro'), top3=(lg.topk(3, 1).indices == ys[:, None]).any(1).float().mean().item())

LG_TE, Y_TE = predict(model, L_te); lab_m = metrics(LG_TE, Y_TE)
print({k: round(v, 4) for k, v in lab_m.items()})
pred = LG_TE.argmax(1).numpy(); y = Y_TE.numpy()
print(classification_report(y, pred, target_names=[c['id'] for c in CLASSES_META], digits=3))
te_df = te.assign(ok=(pred == y))
display(te_df.groupby('crop').ok.mean().round(4).rename('test accuracy').to_frame().T)
cm = confusion_matrix(y, pred, normalize='true')
plt.figure(figsize=(11, 10)); plt.imshow(cm, cmap='Greens'); plt.colorbar()
plt.xticks(range(NC), [c['id'].replace('___', ' ')[:24] for c in CLASSES_META], rotation=90, fontsize=6); plt.yticks(range(NC), [c['id'].replace('___', ' ')[:24] for c in CLASSES_META], fontsize=6)
plt.title('Confusion matrix (row-normalised), test split'); plt.tight_layout(); plt.savefig(OUT / 'confusion_matrix.png', dpi=150); plt.show()
conf_pairs = sorted(((cm[i, j], CLASSES_META[i]['id'], CLASSES_META[j]['id']) for i in range(NC) for j in range(NC) if i != j and cm[i, j] > 0.02), reverse=True)[:10]
print('most confused pairs (true -> predicted, share):'); [print(f'  {a:.2f}  {b}  ->  {c}') for a, b, c in conf_pairs]

## 7. Calibration, confidence threshold and unknown-image gate

In [ ]:
def ece(probs, labels, bins=15):
    conf, pred = probs.max(1); acc = (pred == labels).float(); e = 0.0; edges = torch.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.float().mean().item() * abs(acc[m].mean().item() - conf[m].mean().item())
    return e
def fit_temperature(lg, ys):
    t = torch.zeros(1, requires_grad=True); opt = torch.optim.LBFGS([t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad(); l = F.cross_entropy(lg / t.exp(), ys); l.backward(); return l
    opt.step(closure); return t.exp().item()
def energy(lg): return -torch.logsumexp(lg, 1)
def auroc(pos, neg):
    s = np.concatenate([pos, neg]); r = s.argsort().argsort() + 1
    return float((r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))
def pick_threshold(conf, ok, target):
    for t in np.round(np.arange(0.3, 0.96, 0.05), 2):
        m = conf >= t
        if m.sum() >= 20 and ok[m].mean() >= target: return float(t)
    return None

LG_VA, Y_VA = predict(model, L_va)
TEMP = fit_temperature(LG_VA, Y_VA)
ece_b, ece_a = ece(F.softmax(LG_TE, 1), Y_TE), ece(F.softmax(LG_TE / TEMP, 1), Y_TE)
print(f'temperature {TEMP:.3f} | ECE {ece_b:.4f} -> {ece_a:.4f}')
pv = F.softmax(LG_VA / TEMP, 1); cv_, pr_ = pv.max(1); okv = (pr_ == Y_VA).numpy()
THR = pick_threshold(cv_.numpy(), okv, TARGET_PRECISION)
E_THR = float(np.percentile(energy(LG_VA).numpy(), 95))
print('confidence threshold for', TARGET_PRECISION, 'precision:', THR, '| energy threshold (keeps 95% of normal photos):', round(E_THR, 3))
rows = []
for t in [0.3, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = cv_.numpy() >= t; rows.append(dict(threshold=t, coverage=m.mean(), precision=okv[m].mean() if m.any() else float('nan')))
display(pd.DataFrame(rows).round(3))
if len(ood_df):
    o = ood_df.sample(min(len(ood_df), 1500), random_state=SEED).assign(y=0)
    e_ood = energy(predict(model, loader(o, tf_eval, False))[0]).numpy(); e_id = energy(LG_VA).numpy()
    print(f'unknown crops {OOD_CROPS}: AUROC {auroc(e_ood, e_id):.3f} | caught {(e_ood > E_THR).mean():.1%} at the energy threshold')

## 8. Real field photos (PlantDoc): the honest test
PlantDoc has field photos for many of the same crops. We never train on its test split. First the lab-vs-field gap is measured, then the model is fine-tuned on PlantDoc *train* (plus a replay sample of your data so it does not forget), and measured again. Skip with `USE_PLANTDOC = False`.

In [ ]:
FIELD_FINAL = None; field_rows = []
PLANTDOC_MAP = {
 'Apple Scab Leaf': 'Apple___Apple_scab', 'Apple leaf': 'Apple___healthy', 'Apple rust leaf': 'Apple___Cedar_apple_rust',
 'Bell_pepper leaf': 'Pepper,_bell___healthy', 'Bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot', 'Blueberry leaf': 'Blueberry___healthy',
 'Cherry leaf': 'Cherry_(including_sour)___healthy', 'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
 'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight', 'Corn rust leaf': 'Corn_(maize)___Common_rust_', 'Peach leaf': 'Peach___healthy',
 'Potato leaf early blight': 'Potato___Early_blight', 'Potato leaf late blight': 'Potato___Late_blight', 'Raspberry leaf': 'Raspberry___healthy',
 'Soyabean leaf': 'Soybean___healthy', 'Squash Powdery mildew leaf': 'Squash___Powdery_mildew', 'Strawberry leaf': 'Strawberry___healthy',
 'Tomato Early blight leaf': 'Tomato___Early_blight', 'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot', 'Tomato leaf': 'Tomato___healthy',
 'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot', 'Tomato leaf late blight': 'Tomato___Late_blight', 'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
 'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato mold leaf': 'Tomato___Leaf_Mold',
 'Tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite', 'grape leaf': 'Grape___healthy', 'grape leaf black rot': 'Grape___Black_rot'}
_nk = lambda s: re.sub(r'[^a-z0-9]', '', s.lower())
PD_LOOKUP = {_nk(k): v for k, v in PLANTDOC_MAP.items()}

def plantdoc_frame(root):
    rows, skipped = [], []
    for d in sorted(p for p in Path(root).iterdir() if p.is_dir()):
        raw = PD_LOOKUP.get(_nk(d.name))
        if raw is None or raw not in CLASS_IDX: skipped.append(d.name); continue
        rows += [(str(f), CLASS_IDX[raw], d.name) for f in d.iterdir() if f.suffix.lower() in EXT]
    return pd.DataFrame(rows, columns=['path', 'y', 'pd_class']), skipped

if USE_PLANTDOC:
    PD = WORK / 'plantdoc'
    if not PD.exists(): subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/pratikkayal/PlantDoc-Dataset', str(PD)], check=True)
    assert (PD / 'train').exists() and (PD / 'test').exists(), os.listdir(PD)
    fd_tr, sk1 = plantdoc_frame(PD / 'train'); fd_te, sk2 = plantdoc_frame(PD / 'test')
    fd_val = fd_tr.sample(frac=0.2, random_state=SEED); fd_tr = fd_tr.drop(fd_val.index)
    print('PlantDoc mapped: train', len(fd_tr), '| field-val', len(fd_val), '| field-test', len(fd_te), '| folders skipped:', sorted(set(sk1 + sk2)))
    L_fte, L_fva = loader(fd_te, tf_eval, False), loader(fd_val, tf_eval, False)
    LG_F0, Y_F = predict(model, L_fte); f0 = metrics(LG_F0, Y_F)
    print(f'LAB test accuracy {lab_m["accuracy"]:.3f}  ->  FIELD test accuracy {f0["accuracy"]:.3f}  (gap {100*(lab_m["accuracy"]-f0["accuracy"]):.1f} points)')
    field_rows.append(dict(model='lab-trained', lab_acc=lab_m['accuracy'], field_acc=f0['accuracy'], field_macro_f1=f0['macro_f1']))

    replay = tr.groupby('y', group_keys=False).apply(lambda g: g.sample(min(len(g), 80), random_state=SEED))
    from torch.utils.data import ConcatDataset
    L_mix = DataLoader(ConcatDataset([DS(fd_tr, tf_train)] * 3 + [DS(replay, tf_train)]), batch_size=BS, shuffle=True, num_workers=WORKERS, pin_memory=True, drop_last=True)
    seed_all(); model_f = timm.create_model(BACKBONE, pretrained=False, num_classes=NC); model_f.load_state_dict(torch.load(OUT / 'lab_model.pt'))
    fit(model_f, L_mix, 1 if QUICK else 4, 5e-5, 'field fine-tune', keep_best=False)
    LG_F1, _ = predict(model_f, L_fte); f1_ = metrics(LG_F1, Y_F); LG_TE2, _ = predict(model_f, L_te); lab2 = metrics(LG_TE2, Y_TE)
    field_rows.append(dict(model='+ field fine-tune', lab_acc=lab2['accuracy'], field_acc=f1_['accuracy'], field_macro_f1=f1_['macro_f1']))
    gap = pd.DataFrame(field_rows).round(4); display(gap); gap.to_csv(OUT / 'lab_vs_field.csv', index=False)
    # choose by FIELD-VAL (never the test split), then calibrate on field-val
    fa = lambda m: (predict(m, L_fva)[0].argmax(1) == predict(m, L_fva)[1]).float().mean().item()
    if fa(model_f) >= fa(model):
        model = model_f; print('using the field fine-tuned model (chosen on field-val)')
    lgv, yv_ = predict(model, L_fva); TEMP = fit_temperature(lgv, yv_)
    pf = F.softmax(lgv / TEMP, 1); cf_, prf = pf.max(1); THR = pick_threshold(cf_.numpy(), (prf == yv_).numpy(), TARGET_PRECISION) or THR
    E_THR = float(np.percentile(energy(lgv).numpy(), 95))
    lgt, yt_ = predict(model, L_fte); pt = F.softmax(lgt / TEMP, 1); ct, prt = pt.max(1)
    print(f'final temperature {TEMP:.3f} | threshold {THR} | field-test ECE {ece(pt, yt_):.4f}')
    print('field-test accuracy by PlantDoc class:'); display(pd.DataFrame({'pd_class': fd_te.pd_class.values, 'ok': (prt == yt_).numpy()}).groupby('pd_class').ok.agg(['mean', 'count']).round(3).T)
else:
    print('PlantDoc skipped.')

## 9. Grad-CAM: where is the model looking?

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    layer = model.stages[-1] if hasattr(model, 'stages') else model.blocks[-1]
    model.to(dev).eval()
    imgs = (fd_te.sample(4, random_state=1).path.tolist() if USE_PLANTDOC else []) + te.sample(4, random_state=1).path.tolist()
    fig, ax = plt.subplots(2, 4, figsize=(14, 7))
    with GradCAM(model=model, target_layers=[layer]) as cam:
        for a, p in zip(ax.flat, imgs):
            im = PadSquare()(Image.open(p).convert('RGB')).resize((IMG, IMG)); x = tf_eval(Image.open(p).convert('RGB')).unsqueeze(0).to(dev)
            g = cam(input_tensor=x)[0]; a.imshow(show_cam_on_image(np.asarray(im) / 255., g, use_rgb=True)); a.axis('off')
            a.set_title(CLASSES_META[model(x).argmax().item()]['id'].replace('___', ' ')[:30], fontsize=8)
    plt.suptitle('Grad-CAM (top: field photos, bottom: lab photos)'); plt.savefig(OUT / 'gradcam.png', dpi=120); plt.show()
except Exception as e:
    print('Grad-CAM skipped for this backbone:', e)

## 10. Export for the CropGuard backend

In [ ]:
import onnx, onnxruntime as ort
def export_onnx(m, dummy, path):
    kw = dict(input_names=['input'], output_names=['logits'], dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17)
    try:
        torch.onnx.export(m, dummy, path, dynamo=False, **kw)      # classic exporter (newer PyTorch defaults to another one)
    except Exception:
        torch.onnx.export(m, dummy, path, **kw)

try: timm.layers.set_exportable(True)
except Exception: pass
model.eval().cpu()
ONNX = OUT / 'model.onnx'
export_onnx(model, torch.randn(1, 3, IMG, IMG), str(ONNX))
onnx.checker.check_model(onnx.load(str(ONNX)))
sess = ort.InferenceSession(str(ONNX), providers=['CPUExecutionProvider']); xb = torch.randn(4, 3, IMG, IMG)
with torch.no_grad(): ref = model(xb).numpy()
print('max |torch - onnx| =', float(np.abs(ref - sess.run(None, {'input': xb.numpy()})[0]).max()))
t0 = time.time(); [sess.run(None, {'input': xb[:1].numpy()}) for _ in range(10)]; print(f'CPU latency {(time.time()-t0)/10*1000:.0f} ms/image')

cfg = {'name': f"cropguard-{BACKBONE.split('.')[0]}-{NC}class", 'onnx': 'model.onnx', 'labels': [c['id'] for c in CLASSES_META], 'classes_file': 'class_names.json',
       'input_size': IMG, 'mean': MEAN, 'std': STD, 'temperature': TEMP, 'confidence_threshold': THR if THR is not None else 0.6,
       'energy_threshold': E_THR, 'tta': True, 'healthy_only_crops': HEALTHY_ONLY}
json.dump(cfg, open(OUT / 'model_config.json', 'w', encoding='utf-8'), indent=2)
json.dump({'version': 3, 'input_size': IMG, 'mean': MEAN, 'std': STD, 'classes': CLASSES_META}, open(OUT / 'class_names.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
summary = dict(backbone=BACKBONE, classes=NC, crops=int(df.crop.nunique()), images=len(df), split='group-aware' if HONEST_SPLIT else 'dataset-provided', dataset_valid_leak_pct=LEAK_PCT,
               test=lab_m, ece_before=ece_b, ece_after=ece_a, temperature=TEMP, confidence_threshold=THR, energy_threshold=E_THR, healthy_only_crops=HEALTHY_ONLY, lab_vs_field=field_rows, seed=SEED)
json.dump(summary, open(OUT / 'metrics.json', 'w', encoding='utf-8'), indent=2, default=float)
(OUT / 'MODEL_CARD.md').write_text(f'''# CropGuard model card (auto-generated)
* Backbone: {BACKBONE} (pretrained, fine-tuned) | classes: {NC} | crops: {df.crop.nunique()}
* Split: {summary['split']}; leak of the dataset's own valid folder: {LEAK_PCT if LEAK_PCT is not None else 'n/a'} %
* Test (held-out): accuracy {lab_m['accuracy']:.4f}, macro-F1 {lab_m['macro_f1']:.4f}, top-3 {lab_m['top3']:.4f}
* Calibration: temperature {TEMP:.3f}, ECE {ece_b:.4f} -> {ece_a:.4f}; confidence threshold {THR}
* Field (PlantDoc) lab-vs-field: {field_rows}
* Crops with only healthy images (disease detection NOT supported): {HEALTHY_ONLY}
* Limits: lab-style training photos, 14 crops, severity not predicted by this model.
''', encoding='utf-8')
zip_path = shutil.make_archive(str(WORK / 'cropguard38_model'), 'zip', OUT)
print(sorted(os.listdir(OUT)))
if IN_COLAB:
    from google.colab import files; files.download(zip_path)
else:
    print('Zip saved at', zip_path)

## 11. Put it into your project (Windows, VS Code terminal at `E:\MINOR\cropguard`, `(.venv)` active)
1. The browser downloaded `cropguard38_model.zip`. Extract it so that this file exists: `ml\models\cropguard38\model.onnx` (together with `model_config.json`, `class_names.json`, `metrics.json`).
2. Switch the backend to it:
```
python ml/switch_model.py cropguard38
python -m uvicorn backend.app.main:app --port 8000
```
3. Check `http://localhost:8000/api/v1/health`: `"simulated": false`, `"classes": 38`. The header chip should say "Live model".
4. Advice entries exist for 15 of the 38 classes (tomato, potato, pepper). The others get generic advice and are labelled so. Create empty entries for experts to fill:
```
python ml/datasets/make_kb_scaffolds.py --classes ml/models/cropguard38/class_names.json
```
5. Judge the model by the **test split and the field test**, not by the training curve. Read `metrics.json` and `MODEL_CARD.md`. A model that was not tested on field photos should not be described as field-ready.